In [19]:
# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import nltk
warnings.filterwarnings('ignore')

# **Step 1 - Loading and Preprocessing**

In [20]:
# Load the Dataset

df = pd.read_csv("/content/nlp_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (5937, 2)


,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


Comment    0
Emotion    0
dtype: int64
Duplicate rows: 0


In [21]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Preserve negation words because they can change the meaning of a sentence
stop_words = set(ENGLISH_STOP_WORDS) - {"no", "nor", "not", "never"}

def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Keep alphabetic words and contractions
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    # Remove stopwords and single-character tokens
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Return the cleaned tokens as a string
    return " ".join(tokens)

# Apply preprocessing to the Comment column
df["Cleaned_Comment"] = df["Comment"].fillna("").apply(preprocess_text)

# Display original and preprocessed text
print(df[["Comment", "Cleaned_Comment", "Emotion"]].head())

                                             Comment  \
0  i seriously hate one subject to death but now ...   
1                 im so full of life i feel appalled   
2  i sit here to write i start to dig out my feel...   
3  ive been really angry with r and i feel like a...   
4  i feel suspicious if there is no one outside l...   

                                     Cleaned_Comment Emotion  
0   seriously hate subject death feel reluctant drop    fear  
1                              im life feel appalled   anger  
2  sit write start dig feelings think afraid acce...    fear  
3    ive really angry feel like idiot trusting place     joy  
4   feel suspicious no outside like rapture happened    fear  


In [22]:
print("Original dataset shape:", df.shape)

print("\nMissing values after cleaning:")
print(df[["Comment", "Emotion", "Cleaned_Comment"]].isnull().sum())

print("\nEmotion distribution:")
print(df["Emotion"].value_counts())

print("\nSample before and after preprocessing:")
display(df[["Comment", "Cleaned_Comment"]].head(10))

Original dataset shape: (5937, 3)

Missing values after cleaning:
Comment            0
Emotion            0
Cleaned_Comment    0
dtype: int64

Emotion distribution:
Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64

Sample before and after preprocessing:


,Comment,Cleaned_Comment
0,i seriously hate one subject to death but now ...,seriously hate subject death feel reluctant drop
1,im so full of life i feel appalled,im life feel appalled
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...
3,ive been really angry with r and i feel like a...,ive really angry feel like idiot trusting place
4,i feel suspicious if there is no one outside l...,feel suspicious no outside like rapture happened
5,i feel jealous becasue i wanted that kind of l...,feel jealous becasue wanted kind love true con...
6,when a friend of mine keeps telling me morbid ...,friend keeps telling morbid things happened dog
7,i finally fell asleep feeling angry useless an...,finally fell asleep feeling angry useless anxiety
8,i feel a bit annoyed and antsy in a good way,feel bit annoyed antsy good way
9,i feel like i ve regained another vital part o...,feel like ve regained vital life living


# **Step 2 - Feature Extraction**

In [23]:
# Feature Extration

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True
)

# Transform the cleaned comments into numerical features
X = tfidf_vectorizer.fit_transform(df["Cleaned_Comment"])

# Target variable
y = df["Emotion"]

# Display the feature matrix dimensions
print("Feature matrix shape:", X.shape)
print("Number of features:", len(tfidf_vectorizer.get_feature_names_out()))

# Display a few extracted features
print(tfidf_vectorizer.get_feature_names_out()[:20])

Feature matrix shape: (5937, 30000)
Number of features: 30000
['aa' 'aa meeting' 'aac' 'aac feeling' 'aaron' 'aaron friends' 'ab'
 'abandon' 'abandon sake' 'abandoned' 'abandoned feel' 'abandoned reality'
 'abandonment' 'abandonment embrace' 'abbigail' 'abbigail apraxia' 'abc'
 'ability' 'abit' 'able']


# **Step 3 - Model Development**

In [24]:
# Model Development

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Input text and target labels
X_text = df["Cleaned_Comment"]
y = df["Emotion"]

# Split into training and testing sets (80:20)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert text into TF-IDF features
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True
)

X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_test = tfidf_vectorizer.transform(X_test_text)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (4749, 30000)
Testing data shape: (1188, 30000)


In [25]:
# Train the Naive Bayes Model

from sklearn.naive_bayes import MultinomialNB

# Initialize the Naive Bayes classifier
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train, y_train)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [26]:
# Train the Support Vector Machine model

from sklearn.svm import LinearSVC

# Initialize the Support Vector Machine classifier
svm_model = LinearSVC(random_state=42)

# Train the model
svm_model.fit(X_train, y_train)

print("Support Vector Machine model trained successfully!")

Support Vector Machine model trained successfully!


In [27]:
# Generate Predictions

# Predict emotions for the test dataset
nb_predictions = nb_model.predict(X_test)
svm_predictions = svm_model.predict(X_test)

print("Naive Bayes predictions:", nb_predictions[:10])
print("SVM predictions:", svm_predictions[:10])

Naive Bayes predictions: ['joy' 'fear' 'anger' 'fear' 'fear' 'anger' 'fear' 'anger' 'anger' 'anger']
SVM predictions: ['joy' 'fear' 'anger' 'fear' 'fear' 'anger' 'fear' 'anger' 'anger' 'anger']


# **Step 4 - Model Comparison**

In [28]:
# Evaluate Both Models

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Calculate evaluation metrics for Naive Bayes
nb_results = {
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, nb_predictions),
    "Precision": precision_score(
        y_test, nb_predictions, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, nb_predictions, average="weighted", zero_division=0
    ),
    "F1-Score": f1_score(
        y_test, nb_predictions, average="weighted", zero_division=0
    )
}

# Calculate evaluation metrics for SVM
svm_results = {
    "Model": "Support Vector Machine",
    "Accuracy": accuracy_score(y_test, svm_predictions),
    "Precision": precision_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    ),
    "F1-Score": f1_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    )
}

# Compare the models
comparison_df = pd.DataFrame([nb_results, svm_results])

print(comparison_df.round(4).to_string(index=False))

                 Model  Accuracy  Precision  Recall  F1-Score
           Naive Bayes    0.9184     0.9200  0.9184    0.9184
Support Vector Machine    0.9478     0.9478  0.9478    0.9478


In [29]:
print("Naive Bayes Classification Report")
print(classification_report(y_test, nb_predictions, zero_division=0))

print("SVM Classification Report")
print(classification_report(y_test, svm_predictions, zero_division=0))

Naive Bayes Classification Report
              precision    recall  f1-score   support

       anger       0.91      0.92      0.92       400
        fear       0.89      0.95      0.92       388
         joy       0.96      0.89      0.92       400

    accuracy                           0.92      1188
   macro avg       0.92      0.92      0.92      1188
weighted avg       0.92      0.92      0.92      1188

SVM Classification Report
              precision    recall  f1-score   support

       anger       0.95      0.93      0.94       400
        fear       0.94      0.95      0.94       388
         joy       0.96      0.96      0.96       400

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188



### Model Comparison and Conclusion

Both Multinomial Naive Bayes and Support Vector Machine (SVM) were evaluated using accuracy, precision, recall, and F1-score. Naive Bayes achieved an accuracy of approximately 92%, whereas SVM achieved approximately 95%.

The SVM model performed better across the overall evaluation metrics and achieved a weighted F1-score of 0.95. It also showed strong performance across all three emotion categories: anger, fear, and joy.

SVM is suitable for text-based emotion classification because it handles high-dimensional TF-IDF features effectively and learns decision boundaries that distinguish between emotion categories. Naive Bayes is computationally efficient and provides a useful baseline model. Based on the experimental results, **SVM is the preferred model for this emotion classification task**.